# Clase 13 — Sesión 2: RDDs (Resilient Distributed Datasets)

Ejercicios prácticos y caso de estudio **MediRed** ejecutados en **Jupyter + Almond** (kernel Scala).

Los ficheros que el enunciado pide crear con PowerShell los generaremos directamente desde Scala con `java.nio.file.Files`, para no salir del notebook.

## 🔧 Inicialización del entorno Spark

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.logging.log4j.{Level, LogManager}
import org.apache.logging.log4j.core.config.Configurator
Configurator.setRootLevel(Level.ERROR)
Configurator.setLevel("org",              Level.ERROR)
Configurator.setLevel("org.apache",       Level.ERROR)
Configurator.setLevel("org.apache.spark", Level.ERROR)
Configurator.setLevel("org.sparkproject", Level.ERROR)
Configurator.setLevel("akka",             Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Dia13-RDDs")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

val sc = spark.sparkContext
Configurator.setRootLevel(Level.ERROR)

println(s"✅ Entorno listo — Spark ${spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 03:29:15 INFO SparkContext: Running Spark version 4.1.1
26/04/28 03:29:15 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/28 03:29:15 INFO SparkContext: Java version 17.0.18+8
26/04/28 03:29:16 INFO ResourceUtils: ==============================================================
26/04/28 03:29:16 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/28 03:29:16 INFO ResourceUtils: ==============================================================
26/04/28 03:29:16 INFO SparkContext: Submitted application: Dia13-RDDs
26/04/28 03:29:16 INFO SecurityManager: Changing view acls to: gre
26/04/28 03:29:16 INFO SecurityManager: Changing modify acls to: gre
26/04/28 03:29:16 INFO SecurityManager: Changing view acls groups to: gre
26/04/28 03:29:16 INFO SecurityManager: Changing modify acls groups to: gre
26/04/28 03:29:16 INFO SecurityManager: SecurityManager: authentication disable

✅ Entorno listo — Spark 4.1.1


import $ivy.$
import $ivy.$
import org.apache.logging.log4j.{Level, LogManager}
import org.apache.logging.log4j.core.config.Configurator
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@4a32f25a
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@6d7b8a20

## 🔹 Ejercicio 1 — Crear RDDs y explorar sus propiedades

**Paso 1 — RDD desde una colección**

In [2]:
val temperaturas = sc.parallelize(
  List(22.5, 19.0, 25.3, 18.7, 30.1, 27.8, 21.4)
)

println(s"Número de elementos:   ${temperaturas.count()}")
println(s"Número de particiones: ${temperaturas.getNumPartitions}")
println(s"Primer elemento:       ${temperaturas.first()}")
println(s"Máxima temperatura:    ${temperaturas.max()}")
println(s"Mínima temperatura:    ${temperaturas.min()}")

Número de elementos:   7
Número de particiones: 16
Primer elemento:       22.5
Máxima temperatura:    30.1
Mínima temperatura:    18.7


temperaturas: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:1

**Paso 2 — Controlar el número de particiones con `glom()`**

In [3]:
val tempParticionado = sc.parallelize(
  List(22.5, 19.0, 25.3, 18.7, 30.1, 27.8, 21.4),
  numSlices = 3
)

println(s"Particiones: ${tempParticionado.getNumPartitions}")

tempParticionado.glom().collect().zipWithIndex.foreach { case (particion, idx) =>
  println(s"  Partición $idx: ${particion.mkString(", ")}")
}

Particiones: 3
  Partición 0: 22.5, 19.0
  Partición 1: 25.3, 18.7
  Partición 2: 30.1, 27.8, 21.4


tempParticionado: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[1] at parallelize at cmd3.sc:1

**Paso 3 — Crear el fichero de ventas y leerlo con `textFile`**

En lugar de PowerShell, generamos el fichero desde Scala.

In [4]:
import java.nio.file.{Files, Paths, StandardOpenOption}
import java.nio.charset.StandardCharsets

val rutaVentas = Paths.get("C:/Curso-Scala/datos/ventas_trimestre.txt")
Files.createDirectories(rutaVentas.getParent)

val contenidoVentas =
  """Madrid,Enero,12500.50
    |Barcelona,Enero,9800.00
    |Valencia,Enero,5400.75
    |Madrid,Febrero,13200.00
    |Barcelona,Febrero,10500.50
    |Valencia,Febrero,4900.25
    |Madrid,Marzo,14800.00
    |""".stripMargin

Files.write(rutaVentas, contenidoVentas.getBytes(StandardCharsets.UTF_8))
println(s"Fichero creado en: $rutaVentas")

Fichero creado en: C:\Curso-Scala\datos\ventas_trimestre.txt


import java.nio.file.{Files, Paths, StandardOpenOption}
import java.nio.charset.StandardCharsets
rutaVentas: java.nio.file.Path = C:\Curso-Scala\datos\ventas_trimestre.txt
res4_3: java.nio.file.Path = C:\Curso-Scala\datos
contenidoVentas: String = """Madrid,Enero,12500.50
Barcelona,Enero,9800.00
Valencia,Enero,5400.75
Madrid,Febrero,13200.00
Barcelona,Febrero,10500.50
Valencia,Febrero,4900.25
Madrid,Marzo,14800.00
"""
res4_5: java.nio.file.Path = C:\Curso-Scala\datos\ventas_trimestre.txt

In [5]:
val ventas = sc.textFile("C:/Curso-Scala/datos/ventas_trimestre.txt")

println(s"Líneas leídas:     ${ventas.count()}")
println(s"Primera línea:     ${ventas.first()}")
println(s"Particiones:       ${ventas.getNumPartitions}")

ventas.take(3).foreach(println)

Líneas leídas:     7
Primera línea:     Madrid,Enero,12500.50
Particiones:       2
Madrid,Enero,12500.50
Barcelona,Enero,9800.00
Valencia,Enero,5400.75


ventas: org.apache.spark.rdd.RDD[String] = C:/Curso-Scala/datos/ventas_trimestre.txt MapPartitionsRDD[4] at textFile at cmd5.sc:1

## 🔹 Ejercicio 2 — Transformaciones básicas

**Transformación 1 — `map`: extraer importes**

In [6]:
val importes = ventas
  .filter(_.trim.nonEmpty)
  .map { linea =>
    val campos = linea.split(",")
    campos(2).toDouble
  }

println(s"Importes extraídos: ${importes.count()}")
println(s"Total ventas:       ${importes.sum()}")
println(s"Media de ventas:    ${importes.mean()}")

Importes extraídos: 7
Total ventas:       71102.0
Media de ventas:    10157.42857142857


importes: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[6] at map at cmd6.sc:3

**Transformación 2 — `filter`: ventas superiores a 10.000 €**

In [7]:
val ventasAltas = ventas
  .filter(_.trim.nonEmpty)
  .filter { linea =>
    val importe = linea.split(",")(2).toDouble
    importe > 10000.0
  }

println(s"Ventas > 10.000 €: ${ventasAltas.count()}")
ventasAltas.collect().foreach(println)

Ventas > 10.000 €: 4
Madrid,Enero,12500.50
Madrid,Febrero,13200.00
Barcelona,Febrero,10500.50
Madrid,Marzo,14800.00


ventasAltas: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[9] at filter at cmd7.sc:3

**Transformación 3 — `map` encadenado al `filter`**

In [8]:
val ciudadesConVentasAltas = ventas
  .filter(_.trim.nonEmpty)
  .filter(_.split(",")(2).toDouble > 10000.0)
  .map(_.split(",")(0))

ciudadesConVentasAltas.collect().foreach(println)

Madrid
Madrid
Barcelona
Madrid


ciudadesConVentasAltas: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[12] at map at cmd8.sc:4

**Transformación 4 — `distinct`: ciudades únicas con ventas altas**

In [9]:
val ciudadesUnicas = ciudadesConVentasAltas.distinct()
ciudadesUnicas.collect().foreach(println)

Barcelona
Madrid


ciudadesUnicas: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[15] at distinct at cmd9.sc:1

**Transformación 5 — `flatMap`: separar cada línea en campos**

In [10]:
val todosLosCampos = ventas
  .filter(_.trim.nonEmpty)
  .flatMap(linea => linea.split(","))

println(s"Total campos: ${todosLosCampos.count()}")
todosLosCampos.take(6).foreach(println)

Total campos: 21
Madrid
Enero
12500.50
Barcelona
Enero
9800.00


todosLosCampos: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[17] at flatMap at cmd10.sc:3

## 🔹 Ejercicio 3 — Ver el linaje con `toDebugString`

In [11]:
val rddBase     = sc.textFile("C:/Curso-Scala/datos/ventas_trimestre.txt")
val rddFiltrado = rddBase.filter(_.contains("Madrid"))
val rddMapeado  = rddFiltrado
  .filter(_.trim.nonEmpty)
  .map(_.split(",")(2).toDouble)

println(rddMapeado.toDebugString)

(2) MapPartitionsRDD[22] at map at cmd11.sc:5 []
 |  MapPartitionsRDD[21] at filter at cmd11.sc:4 []
 |  MapPartitionsRDD[20] at filter at cmd11.sc:2 []
 |  C:/Curso-Scala/datos/ventas_trimestre.txt MapPartitionsRDD[19] at textFile at cmd11.sc:1 []
 |  C:/Curso-Scala/datos/ventas_trimestre.txt HadoopRDD[18] at textFile at cmd11.sc:1 []


rddBase: org.apache.spark.rdd.RDD[String] = C:/Curso-Scala/datos/ventas_trimestre.txt MapPartitionsRDD[19] at textFile at cmd11.sc:1
rddFiltrado: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[20] at filter at cmd11.sc:2
rddMapeado: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[22] at map at cmd11.sc:5

## 🔹 Ejercicio 4 — Persistencia: midiendo el impacto de `cache()`

In [12]:
import org.apache.spark.storage.StorageLevel

val rddPesado = sc.parallelize(1 to 500000)
  .map(x => x * x)
  .filter(x => x % 3 == 0)
  .map(x => x.toString + "_procesado")

// --- SIN cache ---
val t0 = System.currentTimeMillis()
rddPesado.count()
val t1 = System.currentTimeMillis()
rddPesado.count()
val t2 = System.currentTimeMillis()

println(s"Sin cache — 1ª ejecución: ${t1 - t0} ms")
println(s"Sin cache — 2ª ejecución: ${t2 - t1} ms")

// --- CON cache ---
rddPesado.cache()

val t3 = System.currentTimeMillis()
rddPesado.count()
val t4 = System.currentTimeMillis()
rddPesado.count()
val t5 = System.currentTimeMillis()

println(s"Con cache  — 1ª ejecución: ${t4 - t3} ms  ← carga en memoria")
println(s"Con cache  — 2ª ejecución: ${t5 - t4} ms  ← desde memoria ✓")

rddPesado.unpersist()
println("Caché liberada.")

Sin cache — 1ª ejecución: 189 ms
Sin cache — 2ª ejecución: 93 ms
Con cache  — 1ª ejecución: 1131 ms  ← carga en memoria
Con cache  — 2ª ejecución: 257 ms  ← desde memoria ✓
Caché liberada.


import org.apache.spark.storage.StorageLevel
rddPesado: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[26] at map at cmd12.sc:6
t0: Long = 1777339959010L
res12_3: Long = 172781L
t1: Long = 1777339959199L
res12_5: Long = 172781L
t2: Long = 1777339959292L
res12_9: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[26] at map at cmd12.sc:6
t3: Long = 1777339959305L
res12_11: Long = 172781L
t4: Long = 1777339960436L
res12_13: Long = 172781L
t5: Long = 1777339960693L
res12_17: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[26] at map at cmd12.sc:6

---
# 🏥 Caso de Estudio — MediRed

Análisis de registros hospitalarios con RDDs.

## 📂 Paso previo — Crear el fichero de ingresos

Generamos el fichero desde Scala (en lugar de PowerShell).

In [13]:
val rutaIngresos = Paths.get("C:/Curso-Scala/medired/ingresos_ayer.txt")
Files.createDirectories(rutaIngresos.getParent)

val contenidoIngresos =
  """P001,HospitalNorte,Cardiologia,programado,48
    |P002,HospitalSur,Urgencias,urgente,0
    |P003,HospitalCentral,Pediatria,programado,24
    |P004,HospitalEste,Traumatologia,urgente,72
    |P005,HospitalNorte,Neurologia,programado,96
    |P006,HospitalOeste,Urgencias,urgente,0
    |P007,HospitalSur,Cardiologia,traslado,120
    |P008,HospitalCentral,Oncologia,programado,168
    |P009,HospitalEste,Urgencias,urgente,0
    |P010,HospitalNorte,Traumatologia,urgente,36
    |P011,HospitalSur,Pediatria,programado,48
    |P012,HospitalOeste,Neurologia,programado,72
    |P013,HospitalCentral,Urgencias,urgente,0
    |P014,HospitalNorte,Cardiologia,programado,60
    |P015,HospitalEste,Oncologia,traslado,144
    |P016,HospitalSur,Urgencias,urgente,6
    |P017,HospitalOeste,Traumatologia,urgente,48
    |P018,HospitalCentral,Cardiologia,programado,84
    |P019,HospitalNorte,Pediatria,programado,24
    |P020,HospitalEste,Neurologia,programado,96
    |P021,HospitalSur,Traumatologia,urgente,12
    |P022,HospitalOeste,Urgencias,urgente,0
    |P023,HospitalCentral,Neurologia,programado,120
    |P024,HospitalNorte,Urgencias,urgente,3
    |P025,HospitalEste,Cardiologia,programado,72
    |P026,HospitalSur,Oncologia,traslado,200
    |P027,HospitalOeste,Pediatria,programado,36
    |P028,HospitalCentral,Traumatologia,urgente,24
    |P029,HospitalNorte,Oncologia,programado,150
    |P030,HospitalEste,Urgencias,urgente,0
    |""".stripMargin

Files.write(rutaIngresos, contenidoIngresos.getBytes(StandardCharsets.UTF_8))
println(s"Fichero creado en: $rutaIngresos")

Fichero creado en: C:\Curso-Scala\medired\ingresos_ayer.txt


rutaIngresos: java.nio.file.Path = C:\Curso-Scala\medired\ingresos_ayer.txt
res13_1: java.nio.file.Path = C:\Curso-Scala\medired
contenidoIngresos: String = """P001,HospitalNorte,Cardiologia,programado,48
P002,HospitalSur,Urgencias,urgente,0
P003,HospitalCentral,Pediatria,programado,24
P004,HospitalEste,Traumatologia,urgente,72
P005,HospitalNorte,Neurologia,programado,96
P006,HospitalOeste,Urgencias,urgente,0
P007,HospitalSur,Cardiologia,traslado,120
P008,HospitalCentral,Oncologia,programado,168
P009,HospitalEste,Urgencias,urgente,0
P010,HospitalNorte,Traumatologia,urgente,36
P011,HospitalSur,Pediatria,programado,48
P012,HospitalOeste,Neurologia,programado,72
P013,HospitalCentral,Urgencias,urgente,0
P014,HospitalNorte,Cardiologia,programado,60
P015,HospitalEste,Oncologia,traslado,144
P016,HospitalSur,Urgencias,urgente,6
P017,HospitalOeste,Traumatologia,urgente,48
P018,HospitalCentral,Cardiologia,programado,84
P019,HospitalNorte,Pediatria,programado,24
P020,HospitalEste,Neurologia,progr

## Tarea 1 — Crear el RDD base y explorar sus propiedades

**Paso 1 — Cargar el fichero**

In [14]:
val ingresos = sc.textFile("C:/Curso-Scala/medired/ingresos_ayer.txt")
  .filter(_.trim.nonEmpty)

println(s"Total de registros cargados: ${ingresos.count()}")
println(s"Número de particiones:       ${ingresos.getNumPartitions}")
println(s"Primer registro:             ${ingresos.first()}")
println("--- Primeros 5 registros ---")
ingresos.take(5).foreach(println)

Total de registros cargados: 30
Número de particiones:       2
Primer registro:             P001,HospitalNorte,Cardiologia,programado,48
--- Primeros 5 registros ---
P001,HospitalNorte,Cardiologia,programado,48
P002,HospitalSur,Urgencias,urgente,0
P003,HospitalCentral,Pediatria,programado,24
P004,HospitalEste,Traumatologia,urgente,72
P005,HospitalNorte,Neurologia,programado,96


ingresos: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[29] at filter at cmd14.sc:2

**Paso 2 — Distribución por particiones (`glom`)**

In [15]:
ingresos.glom().collect().zipWithIndex.foreach { case (particion, idx) =>
  if (particion.nonEmpty) {
    println(s"Partición $idx → ${particion.length} registros")
    println(s"  Primero: ${particion.head}")
    println(s"  Último:  ${particion.last}")
  } else {
    println(s"Partición $idx → vacía")
  }
}

Partición 0 → 16 registros
  Primero: P001,HospitalNorte,Cardiologia,programado,48
  Último:  P016,HospitalSur,Urgencias,urgente,6
Partición 1 → 14 registros
  Primero: P017,HospitalOeste,Traumatologia,urgente,48
  Último:  P030,HospitalEste,Urgencias,urgente,0


**Paso 3 — Forzar más particiones**

In [16]:
val ingresosConParticiones = sc.textFile(
  "C:/Curso-Scala/medired/ingresos_ayer.txt",
  minPartitions = 4
)
println(s"Particiones con minPartitions=4: ${ingresosConParticiones.getNumPartitions}")

Particiones con minPartitions=4: 4


ingresosConParticiones: org.apache.spark.rdd.RDD[String] = C:/Curso-Scala/medired/ingresos_ayer.txt MapPartitionsRDD[32] at textFile at cmd16.sc:3

### 📝 Respuestas — Tarea 1

1. **2 particiones** significan que Spark dividirá el trabajo en **2 tasks paralelas**. En un clúster real, cada partición podría procesarse en un nodo distinto leyendo simultáneamente desde HDFS o el almacenamiento distribuido. A más particiones, mayor paralelismo (siempre que haya cores disponibles).
2. El **linaje** es la cadena de transformaciones registrada por Spark para construir el RDD: `HadoopRDD → MapPartitionsRDD (textFile) → FilteredRDD`. Si una partición se pierde, Spark **no rehace todo**: con el linaje sabe qué bloque del fichero releer y qué transformaciones aplicar para reconstruirla. Esto es la **resiliencia** de los RDDs (la *R* de RDD).
3. `count()` es una **acción** porque devuelve un valor concreto al Driver (un `Long`) y dispara la ejecución del DAG. `filter(...)` es una **transformación** porque devuelve un **nuevo RDD** (lazy) sin ejecutar nada: solo añade un nodo al plan.

## Tarea 2 — Transformaciones encadenadas: indicadores del informe

**Indicador 1 — Total de ingresos urgentes**

In [17]:
val ingresosUrgentes = ingresos.filter { linea =>
  linea.split(",")(3) == "urgente"
}

val totalUrgentes = ingresosUrgentes.count()
println(s"Ingresos urgentes: $totalUrgentes")

Ingresos urgentes: 13


ingresosUrgentes: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[33] at filter at cmd17.sc:1
totalUrgentes: Long = 13L

**Indicador 2 — Pacientes dados de alta el mismo día**

In [18]:
val altaMismoDia = ingresos.filter { linea =>
  val campos = linea.split(",")
  campos(4).toInt == 0
}

println(s"Altas el mismo día: ${altaMismoDia.count()}")

val idsPacientesAlta = altaMismoDia.map(_.split(",")(0))
println("IDs de pacientes con alta el mismo día:")
idsPacientesAlta.collect().foreach(id => print(s"$id "))
println()

Altas el mismo día: 6
IDs de pacientes con alta el mismo día:
P002 P006 P009 P013 P022 P030 


altaMismoDia: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[34] at filter at cmd18.sc:1
idsPacientesAlta: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[35] at map at cmd18.sc:8

**Indicador 3 — Especialidades únicas**

In [19]:
val especialidades = ingresos
  .map(_.split(",")(2))
  .distinct()

println(s"Número de especialidades: ${especialidades.count()}")
println("Especialidades:")
especialidades.collect().sorted.foreach(e => println(s"  - $e"))

Número de especialidades: 6
Especialidades:
  - Cardiologia
  - Neurologia
  - Oncologia
  - Pediatria
  - Traumatologia
  - Urgencias


especialidades: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[39] at distinct at cmd19.sc:2

**Indicador 4 — Hospitales con ingresos por traslado**

In [ ]:
val hospitalesConTraslado = ingresos
  .filter(_.split(",")(3) == "traslado")
  .map(_.split(",")(1))
  .distinct()

println("Hospitales con ingresos por traslado:")
hospitalesConTraslado.collect().sorted.foreach(h => println(s"  - $h"))

### 📝 Respuestas — Tarea 2

1. **Sin `cache()`, Spark recorre el fichero 2 veces** sobre `altaMismoDia`. Cada acción (`count` y `collect` sobre `idsPacientesAlta`) ejecuta el linaje completo desde `textFile` hacia adelante. Los RDDs **no almacenan datos** entre acciones, solo el plan.
2. `distinct()` necesita comparar elementos de **todas las particiones** para descartar duplicados. Para ello Spark **redistribuye** los datos por hash del valor (cada valor cae siempre en la misma partición destino): eso es el **shuffle**. En la Spark UI aparece como un **stage adicional** con datos en las columnas *Shuffle Write* / *Shuffle Read*.
3. El RDD `ingresos` **no cambia**. Los RDDs son **inmutables**: cada transformación produce un RDD nuevo que apunta a su padre. `ingresos` sigue intacto y puede reutilizarse para otras consultas; solo se ha añadido un nuevo nodo (`MapPartitionsRDD`) al grafo de linaje.

## Tarea 3 — Verificar el linaje con `toDebugString`

**Paso 1 — Linaje simple (sin shuffle)**

In [20]:
val ingresosUrgentesHospital = ingresos
  .filter(_.split(",")(3) == "urgente")
  .map(_.split(",")(1))

println("=== Linaje de ingresosUrgentesHospital ===")
println(ingresosUrgentesHospital.toDebugString)

=== Linaje de ingresosUrgentesHospital ===
(2) MapPartitionsRDD[41] at map at cmd20.sc:3 []
 |  MapPartitionsRDD[40] at filter at cmd20.sc:2 []
 |  MapPartitionsRDD[29] at filter at cmd14.sc:2 []
 |  C:/Curso-Scala/medired/ingresos_ayer.txt MapPartitionsRDD[28] at textFile at cmd14.sc:1 []
 |  C:/Curso-Scala/medired/ingresos_ayer.txt HadoopRDD[27] at textFile at cmd14.sc:1 []


ingresosUrgentesHospital: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[41] at map at cmd20.sc:3

**Paso 2 — Linaje con shuffle (`distinct`)**

In [21]:
val especialidadesUnicas = ingresos
  .map(_.split(",")(2))
  .distinct()

println("=== Linaje de especialidadesUnicas ===")
println(especialidadesUnicas.toDebugString)

=== Linaje de especialidadesUnicas ===
(2) MapPartitionsRDD[45] at distinct at cmd21.sc:2 []
 |  ShuffledRDD[44] at distinct at cmd21.sc:2 []
 +-(2) MapPartitionsRDD[43] at distinct at cmd21.sc:2 []
    |  MapPartitionsRDD[42] at map at cmd21.sc:2 []
    |  MapPartitionsRDD[29] at filter at cmd14.sc:2 []
    |  C:/Curso-Scala/medired/ingresos_ayer.txt MapPartitionsRDD[28] at textFile at cmd14.sc:1 []
    |  C:/Curso-Scala/medired/ingresos_ayer.txt HadoopRDD[27] at textFile at cmd14.sc:1 []


especialidadesUnicas: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[45] at distinct at cmd21.sc:2

**Paso 3 — Predicción y verificación**

Predicción (mental, leyendo de abajo hacia arriba):

```
ShuffledRDD             ← distinct
MapPartitionsRDD        ← map (extrae hospital)
FilteredRDD             ← filter programado
FilteredRDD             ← filter nonEmpty (de ingresos)
MapPartitionsRDD        ← textFile
HadoopRDD               ← textFile
```

In [22]:
val resultado = ingresos
  .filter(_.split(",")(3) == "programado")
  .map(_.split(",")(1))
  .distinct()

println(resultado.toDebugString)

(2) MapPartitionsRDD[50] at distinct at cmd22.sc:3 []
 |  ShuffledRDD[49] at distinct at cmd22.sc:3 []
 +-(2) MapPartitionsRDD[48] at distinct at cmd22.sc:3 []
    |  MapPartitionsRDD[47] at map at cmd22.sc:3 []
    |  MapPartitionsRDD[46] at filter at cmd22.sc:2 []
    |  MapPartitionsRDD[29] at filter at cmd14.sc:2 []
    |  C:/Curso-Scala/medired/ingresos_ayer.txt MapPartitionsRDD[28] at textFile at cmd14.sc:1 []
    |  C:/Curso-Scala/medired/ingresos_ayer.txt HadoopRDD[27] at textFile at cmd14.sc:1 []


resultado: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[50] at distinct at cmd22.sc:3

### 📝 Respuestas — Tarea 3

1. Cada nivel de `toDebugString` (leído **de abajo hacia arriba**) representa una **transformación** registrada por Spark: la fuente (`HadoopRDD`), la lectura (`textFile → MapPartitionsRDD`), y luego cada `filter`, `map`, etc. El número entre paréntesis al inicio (p. ej. `(2)`) indica el **número de particiones** del RDD en ese punto del linaje.
2. La diferencia clave es la aparición de un `ShuffledRDD` en el Paso 2 (por culpa del `distinct`). En el Paso 1 todas las transformaciones son **narrow** (cada partición se procesa de forma independiente) → **un único stage**. En el Paso 2 hay una dependencia **wide** (cada partición de salida depende de todas las de entrada) → Spark **corta** el plan en ese punto y crea un **stage adicional**, con un *shuffle* de por medio.

## Tarea 4 — Persistencia: el mismo RDD, varias consultas

**Paso 1 — RDD base limpio y parseado**

In [23]:
val ingresosLimpios = ingresos
  .filter(_.split(",").length == 5)
  .map { linea =>
    val c = linea.split(",")
    (c(0), c(1), c(2), c(3), c(4).toInt)
  }

println(s"Registros limpios: ${ingresosLimpios.count()}")

Registros limpios: 30


ingresosLimpios: org.apache.spark.rdd.RDD[(String, String, String, String, Int)] = MapPartitionsRDD[52] at map at cmd23.sc:3

**Paso 2 — Tres consultas SIN `cache`**

In [24]:
val t0 = System.currentTimeMillis()
val q1 = ingresosLimpios.filter(_._4 == "urgente").count()
val t1 = System.currentTimeMillis()

val q2 = ingresosLimpios.filter(_._5 == 0).count()
val t2 = System.currentTimeMillis()

val q3 = ingresosLimpios.map(_._2).distinct().count()
val t3 = System.currentTimeMillis()

println("--- Sin cache ---")
println(s"  Consulta 1 (urgentes):          $q1 → ${t1 - t0} ms")
println(s"  Consulta 2 (alta mismo día):    $q2 → ${t2 - t1} ms")
println(s"  Consulta 3 (hospitales únicos): $q3 → ${t3 - t2} ms")
println(s"  Tiempo total: ${t3 - t0} ms")

--- Sin cache ---
  Consulta 1 (urgentes):          13 → 37 ms
  Consulta 2 (alta mismo día):    6 → 32 ms
  Consulta 3 (hospitales únicos): 5 → 104 ms
  Tiempo total: 173 ms


t0: Long = 1777340012024L
q1: Long = 13L
t1: Long = 1777340012061L
q2: Long = 6L
t2: Long = 1777340012093L
q3: Long = 5L
t3: Long = 1777340012197L

**Paso 3 — Tres consultas CON `cache`**

In [25]:
ingresosLimpios.cache()

val t4 = System.currentTimeMillis()
val r1 = ingresosLimpios.filter(_._4 == "urgente").count()
val t5 = System.currentTimeMillis()

val r2 = ingresosLimpios.filter(_._5 == 0).count()
val t6 = System.currentTimeMillis()

val r3 = ingresosLimpios.map(_._2).distinct().count()
val t7 = System.currentTimeMillis()

println("--- Con cache ---")
println(s"  Consulta 1 (urgentes):          $r1 → ${t5 - t4} ms  ← carga en memoria")
println(s"  Consulta 2 (alta mismo día):    $r2 → ${t6 - t5} ms  ← desde memoria")
println(s"  Consulta 3 (hospitales únicos): $r3 → ${t7 - t6} ms  ← desde memoria")
println(s"  Tiempo total: ${t7 - t4} ms")

ingresosLimpios.unpersist()
println("Caché liberada.")

--- Con cache ---
  Consulta 1 (urgentes):          13 → 44 ms  ← carga en memoria
  Consulta 2 (alta mismo día):    6 → 29 ms  ← desde memoria
  Consulta 3 (hospitales únicos): 5 → 111 ms  ← desde memoria
  Tiempo total: 184 ms
Caché liberada.


res25_0: org.apache.spark.rdd.RDD[(String, String, String, String, Int)] = MapPartitionsRDD[52] at map at cmd23.sc:3
t4: Long = 1777340014573L
r1: Long = 13L
t5: Long = 1777340014617L
r2: Long = 6L
t6: Long = 1777340014646L
r3: Long = 5L
t7: Long = 1777340014757L
res25_13: org.apache.spark.rdd.RDD[(String, String, String, String, Int)] = MapPartitionsRDD[52] at map at cmd23.sc:3

### 📝 Respuestas — Tarea 4

1. La **primera ejecución con `cache()` no es más rápida** porque Spark, además de calcular el resultado, está **materializando el RDD en memoria** (serializa los registros y los guarda en los bloques de Storage). Ese trabajo extra compensa de sobra a partir de la segunda acción, que ya lee directamente de RAM y se salta todo el linaje.
2. **No tiene sentido cachear todo.** `cache()` consume memoria, y si no hay suficiente Spark expulsa bloques (o los recalcula). La regla es:
   - ✅ Cachea cuando un RDD se va a usar **2 o más veces** y su linaje es **caro** (lecturas, parseo, shuffles previos).
   - ❌ No caches RDDs intermedios que solo se usan una vez: el linaje los reconstruye sin coste extra y ahorras memoria.
3. Si los datos no caben en RAM, en lugar de `cache()` (que equivale a `MEMORY_ONLY`) usaría **`persist(StorageLevel.MEMORY_AND_DISK)`**: lo que entra en RAM se mantiene allí, y las particiones que no quepan se vuelcan a disco. Así evito que Spark **recalcule** las particiones expulsadas, a cambio de un acceso a disco más lento que la RAM (pero más rápido que rehacer el linaje completo).

---
## 🛑 Cierre de la sesión

> Ejecuta esta celda **solo al terminar** todos los ejercicios para liberar los recursos.

In [26]:
spark.stop()
println("Sesión Spark detenida.")

Sesión Spark detenida.
